# 01 - Load and Understand Raw Data

## Objective

Before building the onboarding funnel or analysing airport supply, let us try to understand the structure and quality of the seven raw datasets.

This notebook focuses on:

- dataset size and grain
- column structure and data types
- timestamps and the extraction cutoff
- categorical values
- key and referential-integrity checks
- basic numerical sanity checks
- data quality issues that could affect later analysis

No cleaning or business conclusions are performed here. The goal is to establish a reliable starting point for the analysis.

## 1. Load the seven raw datasets

The assignment provides five datasets for captain onboarding and two datasets for airport supply.

The data was extracted at **2026-06-30 23:59 IST**, so that cutoff will be important when interpreting recent cohorts and downstream activity.

In [1]:
import pandas as pd

In [2]:
captains = pd.read_csv("../data/captains.csv")
captains.head(5)

,captain_id,signup_ts,city,vehicle_type,acquisition_channel,signup_zone_id,device_tier,app_language,age_band
0,CPT104359,2026-01-01 06:08:00,Pune,Auto,organic_app,PUN-Z23,low,kn,25-34
1,CPT120008,2026-01-01 06:52:00,Pune,Cab,referral,PUN-Z05,mid,mr,45+
2,CPT102049,2026-01-01 07:02:00,Hyderabad,Cab,organic_app,HYD-Z01,high,te,35-44
3,CPT123006,2026-01-01 07:15:00,Pune,Cab,organic_app,PUN-Z11,high,hi,25-34
4,CPT102642,2026-01-01 07:27:00,Bangalore,ERickshaw,gc_telecalling,BAN-Z06,low,hi,18-24


In [3]:
doc_events = pd.read_csv("../data/doc_events.csv")
doc_events.head(5)

,event_id,captain_id,doc_type,attempt_no,event_type,event_ts,failure_reason
0,EV000000000,CPT104359,DL,1,upload_success,2026-01-01 20:30:06.229672479,NaN
1,EV000000001,CPT104359,DL,1,verification_pass,2026-01-01 23:26:54.472703731,NaN
2,EV000000002,CPT104359,RC,1,upload_success,2026-01-02 09:36:49.336302871,NaN
3,EV000000003,CPT104359,RC,1,verification_pass,2026-01-02 12:41:51.324092998,NaN
4,EV000000004,CPT104359,AADHAAR,1,upload_success,2026-01-02 14:06:20.401070690,NaN


In [4]:
approvals = pd.read_csv("../data/approvals.csv")
approvals.head(5)

,captain_id,decision_ts,final_status,last_stage_reached,docs_cleared
0,CPT104359,2026-01-06 08:11:48.775157583,approved,NaN,6
1,CPT120008,NaN,dropped_in_docs,AADHAAR,2
2,CPT102049,NaN,dropped_in_docs,PERMIT,3
3,CPT123006,NaN,dropped_in_docs,INSURANCE,5
4,CPT102642,NaN,dropped_in_docs,PERMIT,3


In [5]:
activation = pd.read_csv("../data/activation.csv")
activation.head(5)

,captain_id,first_order_ts,orders_d7,orders_d30,online_hours_d30
0,CPT104359,2026-01-11 12:11:54.524199600,7.0,33.0,26.9
1,CPT105916,2026-01-08 04:16:11.743250188,15.0,48.0,29.1
2,CPT115782,2026-01-10 11:14:57.939110321,9.0,31.0,20.2
3,CPT124093,2026-01-09 16:35:45.322422039,8.0,13.0,11.4
4,CPT101207,2026-01-09 18:43:22.114690071,4.0,9.0,6.1


In [6]:
nudges = pd.read_csv("../data/nudges.csv")
nudges.head(5)

,captain_id,campaign_id,channel,sent_ts,delivered,clicked
0,CPT104359,CAMP_SMS_004,sms,2026-01-02 01:31:34.373244464,1,0
1,CPT102049,CAMP_WA_002,whatsapp,2026-01-02 20:02:31.634258914,1,0
2,CPT123006,CAMP_WA_002,whatsapp,2026-01-03 11:32:05.154545929,1,1
3,CPT105916,CAMP_WA_002,whatsapp,2026-01-05 01:25:34.559467887,1,1
4,CPT107181,CAMP_WA_001,whatsapp,2026-01-01 23:10:49.644254087,1,0


In [7]:
airport_hourly = pd.read_csv("../data/airport_hourly.csv")
airport_hourly.head(5)

,zone_id,zone_type,hour_ts,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,avg_surge_multiplier
0,APT-T1,airport_terminal,2026-05-01 00:00:00,83,31,52,15,9.13,2.04
1,APT-T1,airport_terminal,2026-05-01 01:00:00,111,27,84,10,10.13,2.17
2,APT-T1,airport_terminal,2026-05-01 02:00:00,86,21,65,13,10.89,2.19
3,APT-T1,airport_terminal,2026-05-01 03:00:00,54,25,29,15,9.27,1.80
4,APT-T1,airport_terminal,2026-05-01 04:00:00,36,34,2,17,4.54,1.08


In [8]:
airport_trips = pd.read_csv("../data/airport_trips.csv")
airport_trips.head(5)

,trip_id,pickup_zone_id,drop_zone_id,drop_zone_type,request_ts,trip_distance_km,captain_cancelled,got_return_fare_within_20min,fare_inr
0,TRP00036169,APT-T1,CBD-02,city_core,2026-05-01 00:00:00,8.18,0,1,148.0
1,TRP00018764,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,21.10,0,0,302.0
2,TRP00027350,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,14.22,0,0,231.0
3,TRP00058214,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,9.81,0,0,207.0
4,TRP00044709,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,18.02,1,0,333.0


In [9]:
datasets = {
    "captains": captains,
    "doc_events": doc_events,
    "approvals": approvals,
    "activation": activation,
    "nudges": nudges,
    "airport_hourly": airport_hourly,
    "airport_trips": airport_trips
}

In [10]:
print(f"Loaded {len(datasets)} datasets.")

Loaded 7 datasets.


In [11]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

captains: (25000, 9)
doc_events: (186282, 7)
approvals: (25000, 5)
activation: (4206, 5)
nudges: (16314, 6)
airport_hourly: (10248, 9)
airport_trips: (60000, 9)


## Dataset Inventory

In [12]:
inventory = pd.DataFrame({
    "dataset": datasets.keys(),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

In [13]:
inventory["rows"] = inventory["rows"].map(lambda x: f"{x:,}")
inventory

,dataset,rows,columns
0,captains,"25,000",9
1,doc_events,"186,282",7
2,approvals,"25,000",5
3,activation,"4,206",5
4,nudges,"16,314",6
5,airport_hourly,"10,248",9
6,airport_trips,"60,000",9


## 2. Schema overview

The datasets operate at different grains. That distinction matters later because event level tables cannot be joined to captain level tables blindly without creating duplicate captain rows.

In [14]:
schema_rows = []

for name, df in datasets.items():
    for column in df.columns:
        schema_rows.append(
            {
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype),
                "missing": int(df[column].isna().sum()),
            }
        )

In [15]:
schema = pd.DataFrame(schema_rows)
display(schema)

,dataset,column,dtype,missing
0,captains,captain_id,object,0
1,captains,signup_ts,object,0
2,captains,city,object,0
3,captains,vehicle_type,object,0
4,captains,acquisition_channel,object,0
5,captains,signup_zone_id,object,0
6,captains,device_tier,object,0
7,captains,app_language,object,0
8,captains,age_band,object,0
9,doc_events,event_id,object,0


In [16]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * len(name))
    print(df.columns.tolist())


CAPTAINS
--------
['captain_id', 'signup_ts', 'city', 'vehicle_type', 'acquisition_channel', 'signup_zone_id', 'device_tier', 'app_language', 'age_band']

DOC_EVENTS
----------
['event_id', 'captain_id', 'doc_type', 'attempt_no', 'event_type', 'event_ts', 'failure_reason']

APPROVALS
---------
['captain_id', 'decision_ts', 'final_status', 'last_stage_reached', 'docs_cleared']

ACTIVATION
----------
['captain_id', 'first_order_ts', 'orders_d7', 'orders_d30', 'online_hours_d30']

NUDGES
------
['captain_id', 'campaign_id', 'channel', 'sent_ts', 'delivered', 'clicked']

AIRPORT_HOURLY
--------------
['zone_id', 'zone_type', 'hour_ts', 'requests', 'fulfilled_requests', 'unfulfilled_requests', 'online_captains', 'avg_eta_min', 'avg_surge_multiplier']

AIRPORT_TRIPS
-------------
['trip_id', 'pickup_zone_id', 'drop_zone_id', 'drop_zone_type', 'request_ts', 'trip_distance_km', 'captain_cancelled', 'got_return_fare_within_20min', 'fare_inr']


## 3. Dataset grain and key checks

Expected grain:

| Dataset | Expected grain |
|---|---|
| `captains` | one row per signup/captain |
| `doc_events` | one row per document event |
| `approvals` | one row per captain |
| `activation` | one row per approved captain |
| `nudges` | one row per captain × campaign exposure |
| `airport_hourly` | one row per zone × hour |
| `airport_trips` | one row per sampled airport trip |

The checks below confirm whether the raw data follows these expectations.

In [17]:
key_checks = {
    "captains": ["captain_id"],
    "doc_events": ["event_id"],
    "approvals": ["captain_id"],
    "activation": ["captain_id"],
    "airport_trips": ["trip_id"],
}

In [18]:
key_rows = []

for name, keys in key_checks.items():
    df = datasets[name]
    unique_keys = df[keys].drop_duplicates().shape[0]
    key_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "unique_keys": unique_keys,
            "duplicate_key_rows": len(df) - unique_keys,
        }
    )

In [19]:
key_summary = pd.DataFrame(key_rows)
key_summary

,dataset,rows,unique_keys,duplicate_key_rows
0,captains,25000,25000,0
1,doc_events,186282,186282,0
2,approvals,25000,25000,0
3,activation,4206,4206,0
4,airport_trips,60000,60000,0


In [20]:
nudge_grain = (
    nudges.groupby(["captain_id", "campaign_id"])
    .size()
    .rename("rows_per_captain_campaign")
)

In [21]:
print("Nudge rows per captain × campaign:")
display(nudge_grain.describe().to_frame().T)

Nudge rows per captain × campaign:


,count,mean,std,min,25%,50%,75%,max
rows_per_captain_campaign,16314.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [22]:
hourly_grain = (
    airport_hourly.groupby(["zone_id", "hour_ts"])
    .size()
    .rename("rows_per_zone_hour")
)

In [23]:
print("\nAirport hourly rows per zone × hour:")
display(hourly_grain.value_counts().sort_index().to_frame("row_count"))


Airport hourly rows per zone × hour:


,row_count
rows_per_zone_hour,
1,10248


## 4. Sample records

A small sample is enough to understand the shape of each table without filling the notebook with repeated full table dumps.

In [24]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head(3))


captains


,captain_id,signup_ts,city,vehicle_type,acquisition_channel,signup_zone_id,device_tier,app_language,age_band
0,CPT104359,2026-01-01 06:08:00,Pune,Auto,organic_app,PUN-Z23,low,kn,25-34
1,CPT120008,2026-01-01 06:52:00,Pune,Cab,referral,PUN-Z05,mid,mr,45+
2,CPT102049,2026-01-01 07:02:00,Hyderabad,Cab,organic_app,HYD-Z01,high,te,35-44



doc_events


,event_id,captain_id,doc_type,attempt_no,event_type,event_ts,failure_reason
0,EV000000000,CPT104359,DL,1,upload_success,2026-01-01 20:30:06.229672479,NaN
1,EV000000001,CPT104359,DL,1,verification_pass,2026-01-01 23:26:54.472703731,NaN
2,EV000000002,CPT104359,RC,1,upload_success,2026-01-02 09:36:49.336302871,NaN



approvals


...

,captain_id,campaign_id,channel,sent_ts,delivered,clicked
0,CPT104359,CAMP_SMS_004,sms,2026-01-02 01:31:34.373244464,1,0
1,CPT102049,CAMP_WA_002,whatsapp,2026-01-02 20:02:31.634258914,1,0
2,CPT123006,CAMP_WA_002,whatsapp,2026-01-03 11:32:05.154545929,1,1



airport_hourly


,zone_id,zone_type,hour_ts,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,avg_surge_multiplier
0,APT-T1,airport_terminal,2026-05-01 00:00:00,83,31,52,15,9.13,2.04
1,APT-T1,airport_terminal,2026-05-01 01:00:00,111,27,84,10,10.13,2.17
2,APT-T1,airport_terminal,2026-05-01 02:00:00,86,21,65,13,10.89,2.19



airport_trips


,trip_id,pickup_zone_id,drop_zone_id,drop_zone_type,request_ts,trip_distance_km,captain_cancelled,got_return_fare_within_20min,fare_inr
0,TRP00036169,APT-T1,CBD-02,city_core,2026-05-01 00:00:00,8.18,0,1,148.0
1,TRP00018764,APT-T1,SUB-07,suburban,2026-05-01 00:00:00,21.10,0,0,302.0
2,TRP00027350,APT-T2,TECH-03,tech_park,2026-05-01 00:00:00,14.22,0,0,231.0


## 5. Categorical values

These are the main categorical dimensions available for later segmentation.

The document sequence from the brief is:

1. DL - all
2. RC - all
3. Aadhaar - all
4. Permit - Auto and Cab
5. Fitness - all
6. Insurance - all

The different applicability of Permit will be important when defining the onboarding funnel.

In [25]:
categorical_columns = {
    "captains": [
        "city",
        "vehicle_type",
        "acquisition_channel",
        "device_tier",
        "app_language",
        "age_band",
    ],
    "doc_events": [
        "doc_type",
        "event_type",
        "failure_reason",
    ],
    "approvals": [
        "final_status",
        "last_stage_reached",
    ],
    "nudges": [
        "campaign_id",
        "channel",
    ],
    "airport_hourly": [
        "zone_type",
    ],
    "airport_trips": [
        "drop_zone_type",
    ],
}

In [26]:
for dataset, columns in categorical_columns.items():
    print(f"\n{dataset.upper()}")
    print("-" * len(dataset))
    for column in columns:
        print(f"\n{column}")
        print(datasets[dataset][column].value_counts(dropna=False))


CAPTAINS
--------

city
city
Hyderabad    7663
Delhi        6452
Pune         5448
Bangalore    5437
Name: count, dtype: int64

vehicle_type
vehicle_type
Auto         11363
Cab           8881
ERickshaw     4756
Name: count, dtype: int64

acquisition_channel
acquisition_channel
organic_app       8418
fos_field         6122
referral          5518
paid_digital      3483
gc_telecalling    1459
Name: count, dtype: int64

device_tier
device_tier
low     11565
mid      9533
high     3902
Name: count, dtype: int64

app_language
app_language
kn    5081
te    5064
mr    4956
en    4952
hi    4947
Name: count, dtype: int64

age_band
age_band
25-34    10397
35-44     7086
18-24     4540
45+       2977
Name: count, dtype: int64

DOC_EVENTS
----------

doc_type
doc_type
DL           50780
RC           48099
AADHAAR      31117
FITNESS      21884
PERMIT       20807
INSURANCE    13595
Name: count, dtype: int64

event_type
event_type
upload_success       93236
verification_pass    73274
verification_fa

## 6. Timestamp coverage

The airport datasets cover **May - June 2026**, while the captain onboarding data covers the broader January June period.

This difference in observation windows means the airport analysis should be treated as a separate marketplace view rather than directly combined with the full onboarding cohort.

In [27]:
timestamp_columns = {
    "captains": ["signup_ts"],
    "doc_events": ["event_ts"],
    "approvals": ["decision_ts"],
    "activation": ["first_order_ts"],
    "nudges": ["sent_ts"],
    "airport_hourly": ["hour_ts"],
    "airport_trips": ["request_ts"],
}

In [28]:
timestamp_summary = []

for dataset, columns in timestamp_columns.items():
    for column in columns:
        values = pd.to_datetime(datasets[dataset][column], errors="coerce")
        timestamp_summary.append(
            {
                "dataset": dataset,
                "column": column,
                "min": values.min(),
                "max": values.max(),
                "null_after_parse": int(values.isna().sum()),
            }
        )

In [29]:
timestamp_summary = pd.DataFrame(timestamp_summary)
timestamp_summary

,dataset,column,min,max,null_after_parse
0,captains,signup_ts,2026-01-01 06:08:00.000000000,2026-06-30 21:57:00.000000000,0
1,doc_events,event_ts,2026-01-01 13:30:50.795752785,2026-06-30 23:56:42.723769018,0
2,approvals,decision_ts,2026-01-05 21:40:24.347261353,2026-06-30 22:26:04.692838376,20359
3,activation,first_order_ts,2026-01-06 11:57:58.804450784,2026-06-30 21:36:06.089121524,102
4,nudges,sent_ts,2026-01-01 23:10:49.644254087,2026-06-30 23:37:42.004031897,0
5,airport_hourly,hour_ts,2026-05-01 00:00:00.000000000,2026-06-30 23:00:00.000000000,0
6,airport_trips,request_ts,2026-05-01 00:00:00.000000000,2026-06-30 23:00:00.000000000,0


## 7. Extraction cutoff check

The assignment states that all seven datasets were extracted at **2026-06-30 23:59 IST**.

I check each timestamped dataset for records after that point rather than silently assuming the cutoff is respected.

In [30]:
cutoff = pd.Timestamp("2026-06-30 23:59:59")

In [31]:
cutoff_checks = []

for dataset, column in {
    "captains": "signup_ts",
    "doc_events": "event_ts",
    "approvals": "decision_ts",
    "activation": "first_order_ts",
    "nudges": "sent_ts",
    "airport_hourly": "hour_ts",
    "airport_trips": "request_ts",
}.items():
    ts = pd.to_datetime(datasets[dataset][column], errors="coerce")
    cutoff_checks.append(
        {
            "dataset": dataset,
            "timestamp_column": column,
            "records_after_cutoff": int((ts > cutoff).sum()),
        }
    )

In [32]:
pd.DataFrame(cutoff_checks)

,dataset,timestamp_column,records_after_cutoff
0,captains,signup_ts,0
1,doc_events,event_ts,0
2,approvals,decision_ts,0
3,activation,first_order_ts,0
4,nudges,sent_ts,0
5,airport_hourly,hour_ts,0
6,airport_trips,request_ts,0


## 8. Basic numerical sanity checks

These checks are deliberately simple. They are intended to catch impossible or internally inconsistent values before analysis.

In [33]:
# Airport request accounting
airport_hourly["fulfillment_check"] = (
    airport_hourly["fulfilled_requests"]
    + airport_hourly["unfulfilled_requests"]
    - airport_hourly["requests"]
)

In [34]:
print("Airport rows where fulfilled + unfulfilled != requests:")
print((airport_hourly["fulfillment_check"] != 0).sum())

Airport rows where fulfilled + unfulfilled != requests:
0


In [35]:
airport_hourly.drop(columns="fulfillment_check", inplace=True)

In [36]:
# Activation ordering
print("\nActivation rows where D7 orders exceed D30 orders:")
display(
    activation.loc[
        activation["orders_d7"] > activation["orders_d30"],
        ["captain_id", "orders_d7", "orders_d30"],
    ]
)


Activation rows where D7 orders exceed D30 orders:


,captain_id,orders_d7,orders_d30
1525,CPT123585,4.0,3.0
1919,CPT123478,4.0,3.0
2048,CPT103607,7.0,6.0
2665,CPT124543,10.0,7.0
2770,CPT112526,6.0,5.0
3087,CPT116344,6.0,5.0
3347,CPT105360,4.0,3.0


In [37]:
# Negative numeric values
print("\nNegative numeric values by dataset:")

for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include="number").columns
    negatives = (df[numeric_cols] < 0).sum()
    negatives = negatives[negatives > 0]

    if len(negatives):
        print(f"\n{name}")
        print(negatives)


Negative numeric values by dataset:


In [38]:
print("Airport trip distance/fare summary:")
display(airport_trips[["trip_distance_km", "fare_inr"]].describe().T)

Airport trip distance/fare summary:


,count,mean,std,min,25%,50%,75%,max
trip_distance_km,60000.0,17.883161,8.117206,1.54,11.82,16.62,22.66,66.73
fare_inr,60000.0,279.349983,111.685797,50.00,197.00,263.00,345.00,931.00


In [39]:
print("\nMissing airport trip distance/fare values:")
display(airport_trips[["trip_distance_km", "fare_inr"]].isna().sum().to_frame("missing").T)


Missing airport trip distance/fare values:


,trip_distance_km,fare_inr
missing,0,0


## 9. Missing values

Missingness is not automatically a data error. Some fields are naturally missing depending on the captain's stage or event type.

The purpose here is to identify where missingness exists so that later analysis can interpret it deliberately.

In [40]:
missing_rows = []

for name, df in datasets.items():
    for column in df.columns:
        count = int(df[column].isna().sum())
        if count:
            missing_rows.append(
                {
                    "dataset": name,
                    "column": column,
                    "missing_count": count,
                    "missing_pct": round(100 * count / len(df), 2),
                }
            )

In [41]:
missing_summary = pd.DataFrame(missing_rows)

if len(missing_summary):
    display(
        missing_summary.sort_values(
            ["dataset", "missing_pct"],
            ascending=[True, False],
        )
    )
else:
    print("No missing values found.")

,dataset,column,missing_count,missing_pct
5,activation,orders_d30,834,19.83
6,activation,online_hours_d30,834,19.83
4,activation,orders_d7,187,4.45
3,activation,first_order_ts,102,2.43
1,approvals,decision_ts,20359,81.44
2,approvals,last_stage_reached,4393,17.57
0,doc_events,failure_reason,166510,89.39


## 10. Referential integrity

The event, approval, activation and nudge tables should only contain captain IDs that exist in `captains.csv`.

Activation is also expected to contain only captains whose terminal onboarding status is approved.

In [42]:
captain_ids = set(captains["captain_id"])

In [43]:
referential_checks = {
    "doc_events → captains": set(doc_events["captain_id"]).issubset(captain_ids),
    "approvals → captains": set(approvals["captain_id"]).issubset(captain_ids),
    "activation → captains": set(activation["captain_id"]).issubset(captain_ids),
    "nudges → captains": set(nudges["captain_id"]).issubset(captain_ids),
}

In [44]:
for check, result in referential_checks.items():
    print(f"{check}: {result}")

approved_ids = set(
    approvals.loc[approvals["final_status"] == "approved", "captain_id"]
)

print(
    "Activation subset of approved captains:",
    set(activation["captain_id"]).issubset(approved_ids),
)

doc_events → captains: True
approvals → captains: True
activation → captains: True
nudges → captains: True
Activation subset of approved captains: True


## 11. Event-level consistency checks

A document failure should have a failure reason, while non-failure events should not.

Similarly, a nudge marked as clicked but not delivered is logically unusual and should be flagged rather than silently corrected.

In [45]:
failure_reason_inconsistency = doc_events[
    (
        doc_events["failure_reason"].notna()
        & (doc_events["event_type"] != "verification_fail")
    )
    |
    (
        doc_events["failure_reason"].isna()
        & (doc_events["event_type"] == "verification_fail")
    )
]

In [46]:
print(
    "Rows with inconsistent failure_reason/event_type:",
    len(failure_reason_inconsistency),
)

Rows with inconsistent failure_reason/event_type: 0


In [47]:
invalid_nudges = nudges[
    (nudges["clicked"] == 1)
    & (nudges["delivered"] == 0)
]

print("Clicks without delivery:", len(invalid_nudges))

Clicks without delivery: 457


In [48]:
invalid_attempts = doc_events[
    ~doc_events["attempt_no"].between(1, 3)
]

print("Document events with attempt_no outside 1–3:", len(invalid_attempts))

Document events with attempt_no outside 1–3: 0


In [49]:
print("\nDocument attempt distribution:")
display(doc_events["attempt_no"].value_counts().sort_index().to_frame("events"))


Document attempt distribution:


,events
attempt_no,
1,162176
2,21863
3,2243


# Initial observations and implications for the next notebook

### Dataset structure

There are seven datasets in total: five covering captain onboarding/activation and two covering airport supply. The datasets have different sizes and grains, so later joins need to be designed around the unit of analysis rather than performed mechanically.

### Onboarding data

`captains` is the signup-level base table. `doc_events` is event-level and can contain many rows per captain because documents can be uploaded and verified across multiple attempts. `approvals` provides the terminal onboarding outcome, while `activation` contains post-approval activity. `nudges` records campaign exposures.

The required document order and the special applicability of Permit need to be respected when constructing the funnel.

### Time coverage

The extraction cutoff is June 30, 2026. The airport datasets cover only May–June, so they will be analysed separately from the January–June onboarding funnel.

Recent onboarding cohorts also need to be treated carefully because newer captains may not have had enough time to reach approval or complete downstream activity.

### Data-quality observations

The checks above flag several items that need to be considered rather than silently repaired:

- `failure_reason` is naturally sparse because it is relevant to verification failures.
- `decision_ts` can be absent for non-terminal approval outcomes.
- Some approved captains do not have a first-order timestamp, which becomes relevant in R2A analysis.
- Some activation records have unusual D7/D30 ordering and should not be ignored when those fields are used.
- Some nudge records show clicks without delivery; these should be treated as a data-quality issue rather than arbitrarily corrected.
- Document attempts should remain within the stated maximum of three.
- Airport request accounting should satisfy `fulfilled + unfulfilled = requests`.
- Referential-integrity checks should hold before building captain-level analysis.

### What comes next

The next notebook should establish the cleaned, analysis-ready tables and explicit cohort/funnel definitions. From there, the analysis can focus on the required business questions:

1. where captains are lost between signup and approval;
2. which leaks are large and actionable;
3. whether `CAMP_WA_002` actually improves onboarding completion;
4. whether airport supply pressure supports targeted captain acquisition.

The aim is to keep the later notebooks focused on decisions and evidence rather than repeating raw-data inspection.